<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/02-rag/01-embeddings-retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embeddings & Retrieval

**Goal:** Choose embeddings, build vector search, and hit the classic similarity pitfalls.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks) — the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client; `requests`, `sentence-transformers`, `numpy` are used by this notebook.
2. **Load your API key.** Get a free key at [console.groq.com](https://console.groq.com/) (no credit card). In Colab, add it via the **key icon** in the left sidebar → **Add new secret**, name it exactly `GROQ_API_KEY`, paste the value, and toggle **Notebook access** on. Running locally instead? Set `GROQ_API_KEY` as an environment variable.

(Full walkthrough and model-picking guidance live in [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" requests sentence-transformers numpy

In [ ]:
from aien import setup

# Loads GROQ_API_KEY (Colab Secrets or local env var) and returns a ready
# Groq client. Pass model=... to override the default; if a call later 404s,
# list available models — see 00-setup/00-environment.ipynb.
client, MODEL = setup()

## The problem: your users don't type keywords

To search documents you first cut them into **chunks** — passage-sized pieces, one per retrievable unit (the next cell explains why that step is unavoidable). With chunks in hand, here's the problem embeddings solve.

Someone asks *"how does a connection get set up?"* — and the chunk that answers it says "the three-way handshake," never the words "set up." Keyword search misses it. That gap — users phrase questions in their own words, documents use their own — is what **embeddings** close: they map text to vectors where *similar meaning* lands *nearby*, so a paraphrase still finds its answer.

That's the goal of this notebook: turn chunks into vectors, search them, and wire the top hits into a generated answer. But embeddings buy semantic matching at a price, and the second half is the price — three pitfalls that will show up in your production logs. The load-bearing rule to carry through: **the generated answer is only ever as good as the chunks retrieval fetched.** Get retrieval wrong and no prompt tweak saves you.

## Setup: corpus and chunks

**Why we can't skip chunking here.** In notebook 00 the "documents" were single sentences — one doc, one retrievable unit, nothing to split. Real documents aren't like that: an RFC in this corpus is tens of thousands of tokens, while the embedding model below (`all-MiniLM-L6-v2`) only reads about the first **256 tokens** of whatever you hand it and silently ignores the rest. Embed a whole RFC and you'd get a vector for its opening boilerplate and nothing else — retrieval would be useless. So each document *must* be cut into passage-sized **chunks** before embedding. That's the forcing function behind chunking, and it's why this notebook can't avoid it.

We use a reasonable **section-aware splitter** (split on numbered headings, cap the size) and treat it as a given here — just enough to get chunks to search. *How* to chunk well, and how much the choice moves retrieval quality, is its own notebook (**03**), placed after this one on purpose: you can't judge a chunking strategy until you've watched retrieval succeed and fail on it, which is exactly what you're about to do.

The download cell is repeated verbatim (each notebook in this repo is self-contained), and the chunker is a compact copy of the one notebook 03 builds up.


In [ ]:
# Download the shared corpus: ten IETF RFCs, cached to data/rfc/.
# Every notebook in this repo that needs the corpus includes this cell --
# self-containment over DRY, so each notebook runs top-to-bottom on its own.
import os
import requests

RFC_NUMBERS = [791, 793, 1035, 2616, 4271, 5321, 6455, 6749, 7540, 9110]
RFC_TITLES = {
    791: 'IP', 793: 'TCP', 1035: 'DNS', 2616: 'HTTP/1.1', 4271: 'BGP',
    5321: 'SMTP', 6455: 'WebSocket', 6749: 'OAuth 2.0', 7540: 'HTTP/2',
    9110: 'HTTP Semantics',
}
DATA_DIR = 'data/rfc'
os.makedirs(DATA_DIR, exist_ok=True)

corpus = {}  # rfc number -> raw text
for num in RFC_NUMBERS:
    path = os.path.join(DATA_DIR, f'rfc{num}.txt')
    if not os.path.exists(path):  # cached: skip the download on re-runs
        resp = requests.get(f'https://www.rfc-editor.org/rfc/rfc{num}.txt', timeout=30)
        resp.raise_for_status()
        with open(path, 'w') as f:
            f.write(resp.text)
    with open(path) as f:
        corpus[num] = f.read()

for num in RFC_NUMBERS:
    print(f'RFC {num:>4}  {RFC_TITLES[num]:<15} {len(corpus[num]):>9,} chars')

In [ ]:
# Compact copy of the cleaning + section-aware chunking. Notebook 03 (chunking)
# is where this is built up and its tradeoffs explained; here it's just a given.
import re

def clean_rfc(text):
    text = text.replace('\f', '\n')
    lines = [l for l in text.split('\n')
             if not re.match(r'^.*\[Page \d+\]\s*$', l)        # page footers
             and not re.match(r'^RFC \d+\s+.*\S+ \d{4}\s*$', l)]  # page headers
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(lines)).strip()

HEADING_RE = re.compile(r'^\d+(?:\.\d+)*\.?\s+\S', re.MULTILINE)

def chunk_paragraphs(text, max_chars=2400):
    paras = [p for p in text.split('\n\n') if p.strip()]
    chunks, cur = [], ''
    for p in paras:
        if cur and len(cur) + len(p) + 2 > max_chars:
            chunks.append(cur)
            cur = p
        else:
            cur = cur + '\n\n' + p if cur else p
    if cur:
        chunks.append(cur)
    return chunks

def chunk_sections(text, max_chars=2400):
    starts = [m.start() for m in HEADING_RE.finditer(text)]
    if not starts:
        return chunk_paragraphs(text, max_chars)
    chunks = [text[:starts[0]].strip()] if text[:starts[0]].strip() else []
    for a, b in zip(starts, starts[1:] + [len(text)]):
        sec = text[a:b].strip()
        if len(sec) <= max_chars:
            chunks.append(sec)
        else:  # long section: paragraph-pack it, carrying the heading along
            heading = sec.split('\n', 1)[0]
            for sub in chunk_paragraphs(sec, max_chars):
                chunks.append(sub if sub.startswith(heading) else heading + '\n' + sub)
    return [c for c in chunks if c]

chunk_texts, chunk_meta = [], []
for num in RFC_NUMBERS:
    for c in chunk_sections(clean_rfc(corpus[num])):
        chunk_texts.append(c)
        chunk_meta.append({'rfc': num, 'title': RFC_TITLES[num]})
print(f'{len(chunk_texts)} chunks across {len(RFC_NUMBERS)} RFCs')

## Embed the chunks

We use `all-MiniLM-L6-v2` from sentence-transformers: it runs locally, it's free, and Colab handles the torch install. Production systems use a hosted embeddings API (Voyage, OpenAI, Cohere) — the mechanics you're learning here are identical, and keeping embeddings free means your API budget goes to generation and evals, where it actually buys you something.

On the free Colab CPU embedding a few thousand chunks takes a couple of minutes; on a T4 GPU runtime it's seconds. Either works.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(chunk_texts, normalize_embeddings=True,
                             show_progress_bar=True)
print(embeddings.shape)  # (n_chunks, 384)

## Brute-force cosine search in numpy

The vectors are normalized, so cosine similarity is just a dot product, and searching the whole corpus is one matrix-vector multiply. That's the entire vector database, in about ten lines.

At learning scale — a few thousand chunks — brute force is not a shortcut, it's *correct*: it's exact, it's milliseconds, and it has no index to tune or go stale. Approximate nearest-neighbor indexes (HNSW, IVF) exist for the 100M-vector regime, where exact search stops being affordable and you start trading recall for latency. That world has its own design space — the [vector database walkthrough](https://www.calm.rocks/resources/prepare-interview/system-design/vector-database-walkthrough/) covers it — but don't reach for it before your corpus does.


Run these and read the top-5 for each — with the scores, not just the titles. For well-phrased questions against a corpus this clean, semantic search looks great: the right RFC dominates, the right sections float up.

Now let's break it three ways. Each pitfall below is a real query pattern that will show up in your production logs, demonstrated live against this corpus. Here they are at a glance, then one section each:

| # | Pitfall | Symptom | Why it happens | Cheapest fix |
|---|---|---|---|---|
| 1 | **similar ≠ relevant** | thematically-close but wrong-intent chunks rank top | embeddings match meaning, not the intent the query left unsaid | scope by metadata; enrich the query with known context |
| 2 | **vocabulary mismatch** | exact identifiers (headers, ports, opcodes) not reliably found | 384 numbers can't preserve exact tokens; siblings collide | add keyword search alongside vectors (notebook 02) |
| 3 | **scores aren't confidence** | an unanswerable query still returns a high top score | nearest-neighbor always returns *something* | use a reranker or an eval, not a score threshold |

## Pitfall 1: similar ≠ relevant

Embeddings measure *semantic similarity*, not *relevance to your intent*. Ask about connection timeouts — meaning, as a web developer would, HTTP request timeouts — and watch what comes back.

Run these and read the top-5 for each — with the scores, not just the titles. For well-phrased questions against a corpus this clean, semantic search looks great: the right RFC dominates, the right sections float up.

Now let's break it three ways. Each pitfall below is a real query pattern that will show up in your production logs, demonstrated live against this corpus.

## Pitfall 1: similar ≠ relevant

Embeddings measure *semantic similarity*, not *relevance to your intent*. Ask about connection timeouts — meaning, as a web developer would, HTTP request timeouts — and watch what comes back.


In [ ]:
show('how long should I wait before timing out a connection?')

Run this and note that TCP material (RFC 793, retransmission timeouts) ranks at or near the top — text that is *about* timeouts and *about* connections, and therefore sits close in embedding space — even though a user asking this in an HTTP context wants RFC 9110/2616 material. The embedding faithfully found similar text. Similar is not the same as relevant: relevance depends on intent the query didn't state.

The fixes are boring and effective: let users scope the search (filter on metadata — we have `chunk_meta` for exactly this), or enrich the query with context you already know ("in HTTP, ..."). Try it:


In [ ]:
show('in HTTP, how long should I wait before timing out a connection?')

## Pitfall 2: vocabulary mismatch — where keyword search wins

Embeddings compress text into 384 numbers. Exact identifiers — port numbers, header field names, opcodes — don't survive that compression with their identity intact: `Sec-WebSocket-Accept` and `Sec-WebSocket-Key` land almost on top of each other in embedding space, though they're different fields with different rules.

Compare vector search against a dumb substring grep for two exact-identifier queries:


In [ ]:
def grep(term, k=5):
    hits = [i for i, t in enumerate(chunk_texts) if term.lower() in t.lower()]
    print(f'grep {term!r}: {len(hits)} chunks contain it, e.g.')
    for i in hits[:k]:
        first_line = chunk_texts[i].strip().split('\n')[0][:72]
        print(f'         RFC {chunk_meta[i]["rfc"]:>4} ({chunk_meta[i]["title"]})  {first_line}')
    print()

show('Retry-After')
grep('Retry-After')

show('Sec-WebSocket-Accept')
grep('Sec-WebSocket-Accept')

Run this and compare. The grep is trivially, perfectly on target: every hit literally contains the identifier. The vector search retrieves *thematically nearby* text — other header fields, other handshake material — and may or may not surface the chunks that actually define `Retry-After` or `Sec-WebSocket-Accept` (as opposed to its sibling `Sec-WebSocket-Key`, which sits almost on top of it in embedding space). For queries that are exact identifiers, a 1970s technique beats the embedding model.

This is not an argument against embeddings; it's an argument that neither tool dominates. Notebook 02 combines them.

## Pitfall 3: scores are not calibrated confidence

It's tempting to treat the cosine score as a confidence: "if the top hit is below 0.5, say I don't know." Watch what happens to score magnitudes across queries — including one whose answer is nowhere in this corpus:


In [ ]:
for q in [
    'How does TCP recover from a lost segment?',        # squarely in-corpus (RFC 793)
    'What port does SMTP use?',                          # in-corpus, terse answer
    'How does Kubernetes schedule pods onto nodes?',     # NOT in this corpus at all
]:
    top_score, i = search(q, 1)[0]
    print(f'{top_score:.3f}  top hit: RFC {chunk_meta[i]["rfc"]} ({chunk_meta[i]["title"]})  <- {q}')

Run this and note that the Kubernetes query — completely unanswerable from ten networking RFCs — still gets a top score in the same general range as the answerable ones. Nearest-neighbor search always returns *something*; the score says "this was the closest chunk," not "this chunk answers the question." Score magnitudes also shift with query length and phrasing, so a threshold tuned on one query distribution quietly misbehaves on another.

If you need an "is this actually relevant?" signal, you need a component trained to produce one (a reranker — notebook 02) or an eval that measures it (section 03 of this repo). The raw cosine number is a ranking device, nothing more.

## Retrieval → generation: closing the loop

RAG's last step is mundane: stuff the top-k chunks into the prompt with citation tags and ask the model to answer from them.


In [ ]:
def answer(question, k=5):
    hits = search(question, k)
    context = '\n\n'.join(
        f'[{n + 1}] (RFC {chunk_meta[i]["rfc"]}, {chunk_meta[i]["title"]})\n{chunk_texts[i][:1500]}'
        for n, (_, i) in enumerate(hits)
    )
    prompt = (
        'Answer the question using only the numbered passages below. '
        'Cite the passages you used like [1] or [2][3].\n\n'
        f'{context}\n\nQuestion: {question}'
    )
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=1024,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.choices[0].message.content

print(answer('What does an HTTP 503 response mean, and which header can tell the client when to retry?'))

Run this and check the citations against the retrieved chunks. When retrieval put the right sections in the prompt, the generation step is easy — the model is good at reading.

That's the asymmetry to internalize: **the answer is only as good as the hits.** Generation cannot cite a chunk that retrieval didn't fetch, and it will do something — usually something fluent — with whatever it *was* given. Every pitfall in this notebook is therefore an answer-quality bug waiting to happen, and notebook 04 is about diagnosing them from the answer side.


## Exercises

1. **Metadata filtering.** Add a `rfc=` parameter to `search()` that restricts results to one RFC (mask the score vector with `chunk_meta`). Re-run the pitfall-1 timeout query filtered to RFC 9110 and confirm the intent problem disappears when the user can scope the search.
2. **Chunker A/B.** Rebuild the index with `chunk_paragraphs` (defined in the setup cell above) instead of `chunk_sections`, and re-run the three `show()` queries at the top. Compare top-5 lists side by side — same corpus, same embedding model, different chunking. This is a preview of notebook 03's whole argument: chunking is the cheapest retrieval-quality lever you have, and here you can already feel it move the results.
3. **Score-gap heuristic.** For each of the three queries in pitfall 3, compute the gap between the top-1 and top-5 scores. Is the gap a better out-of-corpus signal than the absolute score? Build a `maybe_unanswerable(query)` function and find a query that fools it.
4. **Query expansion.** Before searching, use the model (`MODEL`, one cheap call) to rewrite the query into three paraphrases, search with all three, and merge results by max score. Test it on the pitfall-2 identifier queries — does expansion rescue vocabulary mismatch, or does keyword search still win?
